# Generating the low resolution (LR) input from original (ground truth) high resolution (HR).


This code defines a function that describes how to turn high-resolution meteorological data into synthetic low-resolution versions.

The function applies the degradation operator 𝐷, which consists of:

1. Gaussian blur (low-pass filter). It removes small-scale details to mimics how a coarse grid model “smooths” fine scales.

2. Coarsening (block mean downsampling by factor=3). It reduces resolution from 2.5 km to 7.5 km by averaging groups of 3×3 pixels.

This produces LR data suitable for training a super-resolution diffusion model.

In [1]:
import xarray as xr
import numpy as np
import dask
import scipy.ndimage as ndi

print('ready')

ready


#### 1. Function defenition

In [ ]:
# -------------------------------------------------------------------
# Degradation operator D
# -------------------------------------------------------------------
def apply_degradation_operator_D(
    ds, 
    factor=3, 
    sigma_px=0.8, 
    vars_intensive=None, 
    yname=None, 
    xname=None
):

    # Detect coordinate names
    if yname is None:
        yname = "south_north" if "south_north" in ds.dims else "y"
    if xname is None:
        xname = "west_east" if "west_east" in ds.dims else "x"
    ds = ds.rename({yname: "y", xname: "x"})

    # Variables to process
    if vars_intensive is None:
        vars_intensive = list(ds.data_vars.keys())

    # --- Gaussian blur + downsampling ---
    def blur_then_blockmean(da):
        def _blur(arr):
            fill = np.where(np.isfinite(arr), arr, np.nanmedian(arr))
            return ndi.gaussian_filter(fill, sigma=sigma_px, mode="nearest")

        blurred = xr.apply_ufunc(
            _blur, da,
            input_core_dims=[["y", "x"]],
            output_core_dims=[["y", "x"]],
            vectorize=True, dask="parallelized"
        )

        return blurred.coarsen(y=factor, x=factor, boundary="trim").mean()

    # Apply operator
    degraded = xr.Dataset()
    for v in vars_intensive:
        degraded[v] = blur_then_blockmean(ds[v])

    # Coarsened coordinates
    degraded = degraded.assign_coords(
        y = ds["y"].coarsen(y=factor, boundary="trim").mean(),
        x = ds["x"].coarsen(x=factor, boundary="trim").mean()
    )

    degraded.attrs["degradation_operator"] = f"Gaussian σ={sigma_px}, factor={factor}"

    return degraded

print("D operator loaded.")

Notes:
   
    ds : xr.Dataset
        High-resolution dataset (2.5 km grid).
    factor : int, default=3
        Downsampling factor (7.5 / 2.5 = 3).
    sigma_px : float, default=0.8
        Gaussian blur radius in pixels. 0.8–1.0 works well.
    vars_intensive : list of str, optional
        List of variable names to process ( ["i10fg", "t2m", "si10"]).
        If None, all variables in ds will be processed.
    yname, xname : str, optional
        Names of spatial dimensions. Automatically detected if None.

 Returns
    
    xr.Dataset
        Coarsened dataset at lower resolution (e.g., 7.5 km).

#### 2. Application of the function.

This code call the function on real high-resolution data. It produces the synthetic 7.5 km LR field and saves LR data to a NetCDF file.

Name of the variables in the dataset:

 i10fg -wind gust
 
 t2m - temperature at 2m
 
 si10 - wind speed at 10m


In [ ]:
# -------------------------------------------------------------------
# Load HR RTMA dataset
# -------------------------------------------------------------------
ds_hr = xr.open_zarr('/network/rit/home/eb954871/basulab/Projects/DFS/DATA/RTMA/RTMA.zarr')

# -------------------------------------------------------------------
# Select only 2024
# -------------------------------------------------------------------
if "time" in ds_hr:
    ds_2024 = ds_hr.sel(time=slice("2024-01-01", "2024-12-31"))
else:
    raise ValueError("Dataset has no 'time' coordinate to filter years.")

print("Selected 2024 data.")

# -------------------------------------------------------------------
# Meteorological variables you want
# -------------------------------------------------------------------
vars_met = ["i10fg", "t2m", "si10"]

# -------------------------------------------------------------------
# Apply degradation operator to 2024
# -------------------------------------------------------------------
ds_lr_2024 = apply_degradation_operator_D(
    ds_2024,
    factor=3,
    sigma_px=0.8,
    vars_intensive=vars_met
)

# -------------------------------------------------------------------
# Save LR dataset
# -------------------------------------------------------------------
ds_lr_2024.to_netcdf("LR_2024.nc")

print("Finished:LR_2024.nc saved.")

#### 3.Murging the files.

This code:
 - Opens all NetCDF files (2018–2024 files)
 - Merges them along the time dimension
 - Writes them as a single Zarr dataset
 - Zarr is chunked, lazy-loading, very efficient for ML parallel=True lets Dask speed it up


In [ ]:
# murging the files with rechunking before saving to Zarr


print("Opening LR files...")
ds_all = xr.open_mfdataset(
    "LR_*.nc",
    combine="by_coords",
    parallel=True,
    chunks={"time": 720, "y": 85, "x": 96}   # <-- force consistent chunking
)

print("Rechunking...")
ds_all = ds_all.chunk({"time": 720, "y": 85, "x": 96})
# 720 is one month (30 days × 24 h); any number is okay as long as consistent.

print("Saving to Zarr...")
ds_all.to_zarr("LR_all_years.zarr", mode="w")

print("Done.")

# Release memory
ds_all.close()



Now, when we have LR data, next step is "preprocessing":

- extract variables from xarray
- stack them as channels
- normalize each variable separately
- possibly crop or pad to match patch size

# Diffusion model

This diffusion model compute residual. This residual field is what the diffusion model learns to generate (difference between HR and LR data). The LR field is used as conditioning. We train Diffusion model on residual patches. Patch size is 64X64. First, model adds Gaussian noise to residual over many timesteps. Then, model learns to predict that noise, conditioned on LR_up.

In [ ]:
import os
import math
import xarray as xr
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
# ==============================
# 1. DATASET: NetCDF/Zarr
# ==============================

class SRDiffusionDataset(Dataset):
    """
    Residual super-resolution dataset:
      - HR: high-resolution target (2.5 km)
      - LR: low-resolution conditioning (7.5 km), same variables

    Steps:
      1) Align HR and LR in time / space.
      2) Bring LR to HR grid by interpolation (xarray).
      3) Compute residual = HR - LR_up (this is x0 for diffusion).
      4) Cut patches of size patch_size x patch_size.
    """
    def __init__(
        self,
        ds_hr: xr.Dataset,
        ds_lr: xr.Dataset,
        vars_main,
        patch_size: int = 64,
        stride: int = 64,
        normalize: bool = True,
        stats: dict | None = None,
    ):
        super().__init__()
        self.vars = list(vars_main)
        self.patch_size = patch_size
        self.stride = stride

        # 1) Align in time and space
        ds_hr, ds_lr = xr.align(ds_hr[self.vars], ds_lr[self.vars], join="inner")

        # 2) Upsample LR to HR grid (so shapes match). Uses y/x dim names.
        ds_lr_up = ds_lr.interp(
            time=ds_hr.time,
            y=ds_hr.y,
            x=ds_hr.x,
            method="linear"
        )

        # 3) To numpy arrays: [T, C, H, W]
        hr = np.stack([ds_hr[v].values for v in self.vars], axis=1)
        lr_up = np.stack([ds_lr_up[v].values for v in self.vars], axis=1)

        # 4) Optionally compute normalization stats on HR
        if stats is None and normalize:
            stats = {}
            for i, v in enumerate(self.vars):
                m = np.nanmean(hr[:, i])
                s = np.nanstd(hr[:, i]) + 1e-6
                stats[v] = {"mean": float(m), "std": float(s)}
        self.stats = stats or {}

        if normalize:
            for i, v in enumerate(self.vars):
                m = self.stats[v]["mean"]
                s = self.stats[v]["std"]
                hr[:, i]    = (hr[:, i]    - m) / s
                lr_up[:, i] = (lr_up[:, i] - m) / s

        # 5) Residual field that diffusion will model
        residual = hr - lr_up  # x0 for diffusion

        self.hr      = hr.astype(np.float32)
        self.lr_up   = lr_up.astype(np.float32)
        self.residual = residual.astype(np.float32)

        # 6) Precompute patch indices
        self.indices = []
        T, C, H, W = self.residual.shape
        ps = self.patch_size
        for t in range(T):
            for y0 in range(0, H - ps + 1, stride):
                for x0 in range(0, W - ps + 1, stride):
                    self.indices.append((t, y0, x0))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t, y0, x0 = self.indices[idx]
        ps = self.patch_size

        res = self.residual[t, :, y0:y0+ps, x0:x0+ps]
        lr  = self.lr_up[t, :, y0:y0+ps, x0:x0+ps]

        return {
            "residual": torch.from_numpy(res),   # clean residual x0
            "lr_up":    torch.from_numpy(lr),    # conditioning
        }

# ==============================
# 2. Simple U-Net backbone
# ==============================

class ResidualSRUNet(nn.Module):
    """
    Small U-Net that predicts noise on the residual field, conditioned on lr_up.
    Input channels = C_residual + C_lr = 2 * n_vars.
    Output channels = C_residual (noise prediction for residual only).
    """
    def __init__(self, in_channels: int, base_channels: int = 64):
        super().__init__()

        self.down1 = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.down2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 2, base_channels * 2, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.bottleneck = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels * 2, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(base_channels * 2, base_channels, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.out = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            # predict noise for residual only (half of input channels)
            nn.Conv2d(base_channels, in_channels // 2, 3, padding=1),
        )

    def forward(self, residual_noisy, lr_up):
        # Concatenate residual and conditioning along channel dim
        x = torch.cat([residual_noisy, lr_up], dim=1)  # [B, 2C, H, W]
        d1 = self.down1(x)
        d2 = self.down2(d1)
        b  = self.bottleneck(d2)
        u1 = self.up1(b)
        # skip connection
        u_cat = torch.cat([u1, d1], dim=1)
        out = self.out(u_cat)
        return out  # predicted noise on residual

# ==============================
# 3. Diffusion utilities (DDPM-style)
# ==============================

class DiffusionConfig:
    def __init__(self, timesteps: int = 1000, beta_start: float = 1e-4, beta_end: float = 0.02):
        self.timesteps = timesteps
        self.beta_start = beta_start
        self.beta_end = beta_end

        self.betas = torch.linspace(beta_start, beta_end, timesteps)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)

        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)

def sample_timesteps(batch_size: int, config: DiffusionConfig, device):
    return torch.randint(0, config.timesteps, (batch_size,), device=device)

def q_sample(x0, t, config: DiffusionConfig, noise=None):
    """
    Forward diffusion: q(x_t | x_0)
    x0: clean residual, [B, C, H, W]
    t: 1D tensor of ints in [0, T-1]
    """
    if noise is None:
        noise = torch.randn_like(x0)

    sqrt_alphas_cumprod_t = config.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)
    sqrt_one_minus_alphas_cumprod_t = config.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)

    return sqrt_alphas_cumprod_t * x0 + sqrt_one_minus_alphas_cumprod_t * noise, noise

# ==============================
# 4. Training loop (template)
# ==============================

def train_diffusion_sr(
    ds_hr_path: str,
    ds_lr_path: str,
    vars_main,
    patch_size: int = 64,
    batch_size: int = 4,
    num_epochs: int = 1,
    lr: float = 1e-4,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
):
    # ---- Load xarray data ----
    ds_hr = xr.open_dataset(ds_hr_path)
    ds_lr = xr.open_dataset(ds_lr_path)

    # ---- Create dataset & dataloader ----
    dataset = SRDiffusionDataset(
        ds_hr=ds_hr,
        ds_lr=ds_lr,
        vars_main=vars_main,
        patch_size=patch_size,
        stride=patch_size,   # non-overlapping patches
        normalize=True,
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    # ---- Model & diffusion config ----
    n_vars = len(vars_main)
    model = ResidualSRUNet(in_channels=2 * n_vars, base_channels=64).to(device)
    config = DiffusionConfig(timesteps=1000)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mse = nn.MSELoss()

    # ---- Training ----
    model.train()
    for epoch in range(num_epochs):
        for batch in loader:
            residual_clean = batch["residual"].to(device)  # x0
            lr_up = batch["lr_up"].to(device)

            b = residual_clean.size(0)
            t = sample_timesteps(b, config, device)

            # forward diffusion
            residual_noisy, noise = q_sample(residual_clean, t, config)

            # predict noise
            pred_noise = model(residual_noisy, lr_up)

            loss = mse(pred_noise, noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {loss.item():.4f}")

    return model, dataset.stats

#### 3. Running the model

This code:

 - loads your HR 2.5-km data

 - loads your LR 7.5-km degraded data

 - extracts the three variables

 - normalizes the data

 - cuts both datasets into many small patches

 - builds your diffusion model architecture

 - trains it for 10 epochs

 - returns the trained model (model) and normalization parameters (stats)

In [ ]:
vars_main = ["i10fg", "t2m", "si10"]

model, stats = train_diffusion_sr(
    ds_hr_path="/network/rit/home/eb954871/basulab/Projects/DFS/DATA/RTMA/RTMA_2p5km.nc",
    ds_lr_path="/network/rit/home/eb954871/basulab/Projects/DFS/DATA/RTMA/LR_7p5km_D_operator.nc",
    vars_main=vars_main,
    patch_size=64,       # your choice 
    batch_size=4,
    num_epochs=10,       # your choice
    lr=1e-4,
)

return model, stats
